In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os
import yaml
import timm

os.chdir("/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/training")

/home/a.nugroho/miniforge3/envs/effort_aigi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Param follows Repo, NOT vice versa

Standard repo structure
1. self.backbone() to extract features
2. self.head as classifier head

## General Function

In [40]:
from detectors import CLIPDetector, ConvNextDetector

cfg_path_clip = "config/detector/clip_ciplab.yaml"
cfg_path_convnext = "config/detector/convnext_xl.yaml"
with open(cfg_path_clip, 'r') as f:    
    config_model = yaml.safe_load(f)

model_now = CLIPDetector(config_model)

with open(cfg_path_convnext, 'r') as f:    
    config_model = yaml.safe_load(f)

model_now = ConvNextDetector(config_model)

In [44]:
# stages => backbone.net.stages
path_weight_ciplab = "/mnt/ssd/workspace/adi/repos/DeepfakeBench/pretrained/clip_ciplab_best.pth"
path_weight_convnext = "/mnt/ssd/workspace/adi/repos/DeepfakeBench/pretrained/convnext_xlarge_384_in22ft1k_30.pth"
load_param = torch.load(path_weight_convnext)

param_dict = {}
for key, value in load_param.items():
    split_key = key.split(".")
    if split_key[0] == "module":
        key = ".".join(split_key[1:])

    # CLIP Ciplab
    if "loss_func" in key:
        continue

    # DFGC Convnext
    if "head" not in key:
        if key.split(".")[0]!="backbone":
            key = ".".join(["backbone"]+key.split("."))
        
    param_dict[key] = value


In [45]:
#level 1
print(set([i.split('.')[0] for i in param_dict.keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in param_dict.keys()]))

{'backbone', 'head'}
{'backbone.stages', 'head.norm', 'head.fc', 'backbone.stem'}


In [46]:
# Model loading
model_now.load_state_dict(param_dict)

<All keys matched successfully>

## CLIP Detector

### Detector Repo

In [22]:
from detectors import CLIPDetector
with open("config/detector/clip_ciplab.yaml", 'r') as f:    
    config_clip = yaml.safe_load(f)

In [23]:
from transformers import AutoProcessor, CLIPModel
model_name="openai/clip-vit-base-patch16"
model = CLIPModel.from_pretrained(model_name)

In [26]:
model_now = CLIPDetector(config_clip)

In [27]:
#level 1
print(set([i.split('.')[0] for i in dict(model_now.named_parameters()).keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters()).keys()]))

print(set(['.'.join(i.split('.')[:3]) for i in dict(model_now.named_parameters()).keys()]))

{'backbone', 'head'}
{'head.bias', 'backbone.encoder', 'backbone.embeddings', 'head.weight', 'backbone.pre_layrnorm', 'backbone.post_layernorm'}
{'backbone.embeddings.patch_embedding', 'backbone.encoder.layers', 'head.bias', 'backbone.pre_layrnorm.bias', 'backbone.embeddings.class_embedding', 'backbone.pre_layrnorm.weight', 'backbone.post_layernorm.bias', 'backbone.post_layernorm.weight', 'head.weight', 'backbone.embeddings.position_embedding'}


### PreTrained model

In [28]:
load_param = torch.load("/mnt/ssd/workspace/adi/repos/DeepfakeBench/pretrained/clip_ciplab_best.pth")

In [29]:
#level 1
print(set([i.split('.')[0] for i in load_param.keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in load_param.keys()]))

{'loss_func', 'backbone', 'head'}
{'loss_func.loss_fn', 'head.bias', 'backbone.encoder', 'backbone.embeddings', 'head.weight', 'backbone.pre_layrnorm', 'backbone.post_layernorm'}


For CLIP - Ciplab, remove loss_func

In [32]:
# stages => backbone.net.stages
param_dict = {}
for key, value in load_param.items():
    if "loss_func" not in key:
        key_new = key
        param_dict[key_new] = value

param_dict.pop("backbone.embeddings.position_ids", None)  # safely remove if exists

tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
         168, 169, 170, 171, 172, 173, 174, 175, 176

In [33]:
model_now.load_state_dict(param_dict)

<All keys matched successfully>

## ConvNext-XL Detector

### Detector Repo

In [45]:
from detectors import ConvNextDetector
with open("config/detector/convnext_xl.yaml", 'r') as f:    
    config_convnext = yaml.safe_load(f)

In [46]:
model_now = ConvNextDetector(config_convnext)

In [47]:
#level 1
print(set([i.split('.')[0] for i in dict(model_now.named_parameters()).keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters()).keys()]))

print(set(['.'.join(i.split('.')[:3]) for i in dict(model_now.named_parameters()).keys()]))

#print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters().keys()]

{'backbone', 'head'}
{'head.norm', 'backbone.stem', 'head.fc', 'backbone.stages'}
{'head.fc.weight', 'backbone.stages.3', 'head.fc.bias', 'backbone.stages.0', 'backbone.stages.1', 'head.norm.bias', 'head.norm.weight', 'backbone.stem.0', 'backbone.stem.1', 'backbone.stages.2'}


In [30]:
# Initialize Backbone
from networks.convnext import ConvNext

model_config = config_convnext['backbone_config']
model_bb = ConvNext(model_config)

In [31]:
from torch import nn
def get_last_linear(module: nn.Module):
    for child in reversed(list(module.modules())):
        if isinstance(child, nn.Linear):
            return child
    return None


In [32]:
# Intialize Backbone
model_name = "convnext_xlarge_384_in22ft1k"
pretrained = False
net = timm.create_model(model_name, pretrained=pretrained)


/home/a.nugroho/miniforge3/envs/effort_aigi/lib/python3.10/site-packages/timm/models/_factory.py:114: UserWarning: Mapping deprecated model name convnext_xlarge_384_in22ft1k to current convnext_xlarge.fb_in22k_ft_in1k_384.
  model = create_fn(


### PreTrained Model

In [33]:
load_param = torch.load("/mnt/ssd/workspace/adi/repos/DeepfakeBench/pretrained/convnext_xlarge_384_in22ft1k_30.pth")

In [34]:
#level 1
print(set([i.split('.')[0] for i in load_param.keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in load_param.keys()]))

{'module'}
{'module.stem', 'module.head', 'module.stages'}


In [35]:
# Remove 'module' prefix
load_param_new = {}
for key, value in load_param.items():
    new_key = ".".join(key.split(".")[1:])
    load_param_new[new_key] = value

In [36]:
#level 1
print(set([i.split('.')[0] for i in load_param_new.keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in load_param_new.keys()]))

{'stem', 'stages', 'head'}
{'stages.2', 'stem.1', 'head.fc', 'stages.1', 'stem.0', 'head.norm', 'stages.0', 'stages.3'}


In [37]:
print(set(['.'.join(i.split('.')[:10]) for i in load_param_new.keys() if "head" in i]))

{'head.norm.bias', 'head.fc.weight', 'head.norm.weight', 'head.fc.bias'}


In [38]:
load_param_new

{'stem.0.weight': tensor([[[[-3.3842e-02, -9.0452e-02, -1.3375e-01, -8.4766e-02],
           [-4.6690e-02, -4.8938e-02, -1.0572e-01, -5.7421e-02],
           [ 6.2109e-02,  8.2406e-03,  1.7057e-02, -4.9751e-03],
           [ 7.5093e-02,  6.6128e-02,  3.6368e-02,  5.5522e-02]],
 
          [[-1.5399e-02,  3.0964e-03, -7.0359e-02, -3.7707e-02],
           [-4.5915e-04, -4.8534e-02,  1.1967e-02, -3.2750e-02],
           [ 7.4857e-03,  3.1838e-02, -2.6867e-02,  2.2719e-02],
           [ 3.6013e-02,  1.4091e-02,  2.9451e-02,  1.1422e-02]],
 
          [[ 5.3450e-02,  1.8138e-01,  1.9925e-01,  1.3097e-01],
           [-8.3630e-03,  1.1366e-01,  1.2033e-01,  1.1212e-01],
           [-7.5506e-02, -2.7751e-02,  7.2494e-03, -3.2360e-03],
           [-1.3299e-01, -1.0018e-01, -8.2630e-02, -6.7995e-02]]],
 
 
         [[[-7.3874e-03, -1.5057e-02,  5.3321e-03, -9.4389e-03],
           [-1.0994e-02,  2.6048e-02, -2.0379e-02,  1.1645e-02],
           [ 6.5645e-03, -1.8141e-02,  2.3801e-02, -7.7878e-0

For DFGC ConvNext-XL, add backbone.net as prefix

In [42]:
# stages => backbone.net.stages
param_dict = {}
for key, value in load_param_new.items():
    if "head" not in key:
        key_new = ".".join(["backbone"]+key.split("."))
        param_dict[key_new] = value
    else:
        key_new = key
        param_dict[key_new] = value


#['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters()).keys()]

In [43]:
param_dict.keys()

#level 1
print(set([i.split('.')[0] for i in param_dict.keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in param_dict.keys()]))

print(set(['.'.join(i.split('.')[:3]) for i in param_dict.keys()]))

#print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters().keys()]

{'backbone', 'head'}
{'head.norm', 'backbone.stem', 'head.fc', 'backbone.stages'}
{'head.fc.weight', 'backbone.stages.3', 'head.fc.bias', 'backbone.stages.0', 'backbone.stages.1', 'head.norm.bias', 'head.norm.weight', 'backbone.stem.0', 'backbone.stem.1', 'backbone.stages.2'}


In [44]:
model_now.load_state_dict(param_dict)

<All keys matched successfully>

## EffortDetector

### Detector Repo

In [5]:
from detectors import EffortDetector
with open("config/detector/effort_vh.yaml", 'r') as f:    
    config_effort = yaml.safe_load(f)

In [6]:
model_now = EffortDetector(config_effort)

In [7]:
#level 1
print(set([i.split('.')[0] for i in dict(model_now.named_parameters()).keys()]))

# level 2
print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters()).keys()]))

print(set(['.'.join(i.split('.')[:3]) for i in dict(model_now.named_parameters()).keys()]))

#print(set(['.'.join(i.split('.')[:2]) for i in dict(model_now.named_parameters().keys()]

{'head', 'backbone'}
{'head.weight', 'head.bias', 'backbone.pre_layrnorm', 'backbone.embeddings', 'backbone.post_layernorm', 'backbone.encoder'}
{'backbone.embeddings.class_embedding', 'backbone.embeddings.patch_embedding', 'head.weight', 'backbone.pre_layrnorm.weight', 'head.bias', 'backbone.post_layernorm.weight', 'backbone.pre_layrnorm.bias', 'backbone.encoder.layers', 'backbone.post_layernorm.bias', 'backbone.embeddings.position_embedding'}
